In [1]:
import torch

print("GPU available:", torch.cuda.is_available())
print("GPU device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

GPU available: True
GPU device: Tesla T4


In [2]:
import os, sys, torch
print(sys.executable)
print("LD_LIBRARY_PATH =", os.environ.get("LD_LIBRARY_PATH"))
print("CUDA_HOME =", os.environ.get("CUDA_HOME"))
print("CUDA_PATH =", os.environ.get("CUDA_PATH"))
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("torch cuda:", torch.version.cuda)
print("cudnn:", torch.backends.cudnn.version())

/home/anushkanayak15/semg-keystroke-decoding/venv_clean/bin/python3
LD_LIBRARY_PATH = None
CUDA_HOME = None
CUDA_PATH = None
torch: 2.3.0+cu121
cuda available: True
torch cuda: 12.1
cudnn: 8902


**Run 1 - Baseline**

Goal: establish the reference CNN+BiLSTM performance with no new preprocessing.

normalization: none

Gaussian noise: 0

amplitude scaling: 0

channel dropout: 0

This answers: How well does the model do before any new signal preprocessing?

In [3]:
!python -m emg2qwerty.train \
  model=cnn_bilstm_ctc \
  transforms=cnn_bilstm_preproc_aug \
  user=single_user \
  trainer.max_epochs=50 \
  trainer.accelerator=gpu \
  trainer.devices=1 \
  normalize.mode=none \
  gaussian_noise.std=0.0 \
  amplitude_scale.scale=0.0 \
  channel_dropout.drop_prob=0.0

[2026-03-10 08:36:21,421][__main__][INFO] - 
Config:
user: single_user
dataset:
  train:
  - user: 89335547
    session: 2021-06-03-1622765527-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-02-1622681518-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-04-1622863166-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627003020-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-21-1626916256-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627004019-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-05-1622885888-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-02-1622679967-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f

**Run 2 — Per-channel z-score normalization**

Goal: test whether channel-wise standardization helps.

normalization: per-channel z-score

Gaussian noise: 0

amplitude scaling: 0

channel dropout: 0

This is likely the strongest normalization because each EMG channel can have its own baseline and variance.

In [5]:
!python -m emg2qwerty.train \
  model=cnn_bilstm_ctc \
  transforms=cnn_bilstm_preproc_aug \
  user=single_user \
  trainer.max_epochs=50 \
  trainer.accelerator=gpu \
  trainer.devices=1 \
  normalize.mode=per_channel_zscore \
  gaussian_noise.std=0.0 \
  amplitude_scale.scale=0.0 \
  channel_dropout.drop_prob=0.0

[2026-03-10 17:09:32,689][__main__][INFO] - 
Config:
user: single_user
dataset:
  train:
  - user: 89335547
    session: 2021-06-03-1622765527-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-02-1622681518-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-04-1622863166-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627003020-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-21-1626916256-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627004019-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-05-1622885888-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-02-1622679967-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f

**Run 3 - Global normalization**

Goal: test whether simple whole-window normalization is enough.

normalization: global

Gaussian noise: 0

amplitude scaling: 0

channel dropout: 0

This makes Run 2 vs Run 3 a clean normalization comparison.

In [10]:
!python -m emg2qwerty.train \
  model=cnn_bilstm_ctc \
  transforms=cnn_bilstm_preproc_aug \
  user=single_user \
  trainer.max_epochs=50 \
  trainer.accelerator=gpu \
  trainer.devices=1 \
  normalize.mode=global \
  gaussian_noise.std=0.0 \
  amplitude_scale.scale=0.0 \
  channel_dropout.drop_prob=0.0

[2026-03-11 10:45:37,336][__main__][INFO] - 
Config:
user: single_user
dataset:
  train:
  - user: 89335547
    session: 2021-06-03-1622765527-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-02-1622681518-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-04-1622863166-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627003020-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-21-1626916256-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627004019-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-05-1622885888-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-02-1622679967-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f

**Run 4 - Best normalization + mild Gaussian noise**

Goal: test robustness to sensor noise.

Use the best normalization from Runs 2 and 3. Most likely that will be per_channel_zscore.

normalization: best one

Gaussian noise: 0.01

amplitude scaling: 0

channel dropout: 0

This checks whether a small amount of additive noise improves generalization.

In [23]:
!python -m emg2qwerty.train \
  model=cnn_bilstm_ctc \
  transforms=cnn_bilstm_preproc_aug \
  user=single_user \
  trainer.max_epochs=50 \
  trainer.accelerator=gpu \
  trainer.devices=1 \
  normalize.mode=per_channel_zscore \
  gaussian_noise.std=0.01 \
  amplitude_scale.scale=0.0 \
  channel_dropout.drop_prob=0.0

[2026-03-13 16:38:07,584][__main__][INFO] - 
Config:
user: single_user
dataset:
  train:
  - user: 89335547
    session: 2021-06-03-1622765527-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-02-1622681518-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-04-1622863166-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627003020-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-21-1626916256-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627004019-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-05-1622885888-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-02-1622679967-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f

**Run 5 — Best normalization + mild amplitude scaling**

Goal: test robustness to signal gain variation.

normalization: best one

Gaussian noise: 0

amplitude scaling: 0.05

channel dropout: 0

This simulates realistic variation in EMG amplitude caused by differences in contact or sensor gain.

In [22]:
!python -m emg2qwerty.train \
  model=cnn_bilstm_ctc \
  transforms=cnn_bilstm_preproc_aug \
  user=single_user \
  trainer.max_epochs=50 \
  trainer.accelerator=gpu \
  trainer.devices=1 \
  normalize.mode=per_channel_zscore \
  gaussian_noise.std=0.0 \
  amplitude_scale.scale=0.05 \
  channel_dropout.drop_prob=0.0

[2026-03-13 05:40:26,459][__main__][INFO] - 
Config:
user: single_user
dataset:
  train:
  - user: 89335547
    session: 2021-06-03-1622765527-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-02-1622681518-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-04-1622863166-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627003020-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-21-1626916256-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627004019-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-05-1622885888-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-02-1622679967-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f

**Run 6 - Final best combined pipeline**

Goal: build one realistic final augmentation pipeline.

normalization: best one

Gaussian noise: 0.01

amplitude scaling: 0.05

channel dropout: 0.10

This final run tells the story:
after finding the best signal standardization, you added small realistic perturbations to improve robustness and generalization.

In [9]:
!python -m emg2qwerty.train \
  model=cnn_bilstm_ctc \
  transforms=cnn_bilstm_preproc_aug \
  user=single_user \
  trainer.max_epochs=50 \
  trainer.accelerator=gpu \
  trainer.devices=1 \
  normalize.mode=per_channel_zscore \
  gaussian_noise.std=0.01 \
  amplitude_scale.scale=0.05 \
  channel_dropout.drop_prob=0.10

[2026-03-11 01:26:45,750][__main__][INFO] - 
Config:
user: single_user
dataset:
  train:
  - user: 89335547
    session: 2021-06-03-1622765527-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-02-1622681518-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-04-1622863166-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627003020-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-21-1626916256-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627004019-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-05-1622885888-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-02-1622679967-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f